# Phase 21: Lấy số liệu để điền các [CHECK] trong bản thảo v11

Notebook này **chỉ đọc** các output đã có trên Drive (không train, không chạy lại mô hình) và in ra các bảng cần điền vào `S2I_manuscript_v11.md` và `S2I_supplement_v11.md`.

| Cell | Điền vào | Nguồn |
|---|---|---|
| 2 | Bảng S1: số trial theo bệnh nhân và mức QC | `data/s2i_v1_noqc`, `s2i_v1`, `s2i_qc_session`, `s2i_qc_calib` |
| 3 | Bảng S3: guard gap theo bệnh nhân | `runs/checks18/gapcheck.md` |
| 4 | Bảng S7: cấu hình của dòng Spearman ở mức secondary | `runs/review11/reliability*.md` |
| 5 | Bảng S8: độ bền theo session | `runs/shared_bip_car/readout_longevity` |
| 6 | Bảng S11a: kết quả theo bệnh nhân, mức primary | các thư mục `readout_*_strict_qc_none` |
| 7 | Bảng S11c: kết quả theo bệnh nhân, ScaleOLDNEW | `runs/p17/*/…_primary` |
| 8 | Giá trị của encoder short (0,203 hay 0,204) | `runs/shared_short`, `shared_no_env`, `shared_bip_car` |

Mỗi cell in kết quả và cũng ghi vào **`runs/checks21/manuscript_checks.md`**. Chạy hết các cell (CPU là đủ), rồi gửi lại file đó.

Mỗi bảng có dòng **"kiểm tra"** so trung bình tính lại với số đã có trong bản thảo. Nếu dòng này báo lệch, đừng điền, hãy gửi lại để tôi xem.

In [ ]:
#@title 1. Mount Drive, đường dẫn, hàm phụ
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json
import numpy as np
ROOT = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
DATA, RUNS = f'{ROOT}/data', f'{ROOT}/runs'
MAIN = f'{RUNS}/shared_bip_car'
OUT = f'{RUNS}/checks21'
os.makedirs(OUT, exist_ok=True)
REPORT = f'{OUT}/manuscript_checks.md'
open(REPORT, 'w', encoding='utf-8').write('# Manuscript checks (Phase 21)\n\n')

def emit(text):
    print(text)
    with open(REPORT, 'a', encoding='utf-8') as fh:
        fh.write(text + '\n')

def load_dir(d):
    """readout_sub-*.json in a folder -> {subject: result}."""
    out = {}
    for f in sorted(glob.glob(f'{d}/readout_sub-*.json')):
        r = json.load(open(f))
        out[r.get('subject', os.path.basename(f)[8:-5])] = r
    return out

def model_r(r, model, frac='0.2'):
    """Explicit-prediction r if present (as in the manuscript), else the sufficient-statistics r."""
    ex = r.get('explicit', {}).get(model)
    if ex is not None and abs(ex.get('frac', 0.2) - float(frac)) < 1e-9:
        return ex['mean']['r'], 'explicit'
    m = r['fracs'].get(frac, {}).get('models', {}).get(model)
    return (m['r'], 'stats') if m else (np.nan, 'missing')

def check(label, value, expected, tol=0.0015):
    ok = np.isfinite(value) and abs(value - expected) <= tol
    emit(f'- kiểm tra {label}: tính lại {value:.4f}, bản thảo {expected:.3f} -> ' + ('KHỚP' if ok else '**LỆCH**'))

for p in [f'{DATA}/s2i_v1_noqc', f'{DATA}/s2i_v1', f'{DATA}/s2i_qc_session', f'{DATA}/s2i_qc_calib', MAIN,
          f'{RUNS}/checks18', f'{RUNS}/review11', f'{RUNS}/p17']:
    print(('OK   ' if os.path.exists(p) else 'THIẾU ') + p)

In [ ]:
#@title 2. Bảng S1: số trial theo bệnh nhân và mức QC (đọc trial_bounds, khoảng 2-5 phút)
import torch
FS = 200
LEVELS = [('Primary (no QC)', 's2i_v1_noqc'), ('Secondary (artefact-screened)', 's2i_v1'),
          ('Secondary, re-derived from no-QC', 's2i_qc_session'), ('Sensitivity (calibration-only)', 's2i_qc_calib')]

def trial_info(path):
    d = torch.load(path, weights_only=False, map_location='cpu')
    b = d['trial_bounds'].numpy()
    n = len(b)
    nc = max(3, int(round(n * 0.2)))
    return n, nc, max(0, n - nc - 1), float((b[:nc, 1] - b[:nc, 0]).sum()) / FS / 60

info = {}
for lab, folder in LEVELS:
    for p in sorted(glob.glob(f'{DATA}/{folder}/sub-[0-9][0-9].pt')):
        info[(folder, os.path.basename(p)[:-3])] = trial_info(p)
subs = sorted({s for _, s in info})
emit('## Table S1 (per patient)\n')
emit('Each cell: kept trials / calibration trials / test trials / calibration minutes.\n')
emit('| Patient | ' + ' | '.join(l for l, _ in LEVELS) + ' |')
emit('|---' * (len(LEVELS) + 1) + '|')
for s in subs:
    cells_ = []
    for _, f in LEVELS:
        v = info.get((f, s))
        cells_.append('–' if v is None else f'{v[0]} / {v[1]} / {v[2]} / {v[3]:.1f}')
    emit(f'| {s} | ' + ' | '.join(cells_) + ' |')
tot = []
for _, f in LEVELS:
    vs = [info[(f, s)] for s in subs if (f, s) in info]
    tot.append(f'{sum(v[0] for v in vs)} trials, mean calib {np.mean([v[3] for v in vs]):.1f} min, {len(vs)} pts' if vs else '–')
emit('| **Total** | ' + ' | '.join(tot) + ' |\n')
for f, exp_n in (('s2i_v1_noqc', 3185), ('s2i_v1', 2302), ('s2i_qc_calib', 1975)):
    n = sum(info[(f, s)][0] for s in subs if (f, s) in info)
    emit(f'- kiểm tra tổng trial {f}: tính lại {n}, bản thảo {exp_n} -> ' + ('KHỚP' if n == exp_n else '**LỆCH**'))
n13 = sum(info[('s2i_qc_calib', s)][0] for s in subs if ('s2i_qc_calib', s) in info and s != 'sub-13')
emit(f'- sensitivity không tính sub-13 (bị loại vì chỉ còn 7 trial): {n13} trial')
emit('- Ghi chú: mức sensitivity loại bệnh nhân 13 khỏi phân tích; bảng vẫn liệt kê nếu file tồn tại.\n')

In [ ]:
#@title 3. Bảng S3: guard gap theo bệnh nhân
p = f'{RUNS}/checks18/gapcheck.md'
emit('## Table S3 (per patient), copied from runs/checks18/gapcheck.md\n')
emit(open(p, encoding='utf-8').read() if os.path.exists(p) else '**THIẾU** runs/checks18/gapcheck.md (chạy lại S2I_Phase18_Checks_Colab.ipynb)')

In [ ]:
#@title 4. Bảng S7: cấu hình của dòng Spearman ở mức secondary (0,83; 0,57-0,97)
emit('## Table S7: reliability outputs (first lines name the readout folder and model)\n')
for f in sorted(glob.glob(f'{RUNS}/review11/reliability*.md')):
    lines = open(f, encoding='utf-8').read().splitlines()
    emit(f'### {os.path.basename(f)}')
    emit('\n'.join(lines[:6]) + '\n')

In [ ]:
#@title 5. Bảng S8: độ bền theo session (readout_longevity, secondary, transductive)
R = load_dir(f'{MAIN}/readout_longevity')
rows = []
for s, r in R.items():
    fr, order = r['fracs']['0.2'], r['session_order']
    last = order.index(fr['calib_sessions'][-1])
    for m in ('base', 'enc', 'combo'):
        for ses, v in fr['models'][m].get('sessions', {}).items():
            if r['test_trials_per_session'].get(ses, 0) >= 5:
                rows.append((s, ses, order.index(ses) - last, m, v, r['test_trials_per_session'][ses]))
emit('## Table S8 (per gap; gap 0 = rest of the calibration session; sessions with >= 5 test trials)\n')
emit('| Gap (sessions) | Patients | Base | Single encoder | Ensemble (combo) | Ensemble − base |')
emit('|---|---|---|---|---|---|')
for g in sorted({x[2] for x in rows}):
    mean = {m: np.mean([x[4] for x in rows if x[2] == g and x[3] == m]) for m in ('base', 'enc', 'combo')}
    n = len({x[0] for x in rows if x[2] == g and x[3] == 'combo'})
    emit(f"| {g} | {n} | {mean['base']:.3f} | {mean['enc']:.3f} | {mean['combo']:.3f} | {mean['combo'] - mean['base']:+.3f} |")
emit(f"\n- patient-sessions (combo): {sum(1 for x in rows if x[3] == 'combo')} (bản thảo: 55 trong mixed model)\n")

In [ ]:
#@title 6. Bảng S11a: theo bệnh nhân, mức primary (no QC, strict, 20 %)
SRC = [('Base', f'{MAIN}/readout_combo5_strict_qc_none', 'base', 0.167),
       ('Fixed bank + LAG9', f'{RUNS}/shared_no_env/readout_fixed_strict_qc_none', 'combo_nosess', 0.192),
       ('One learned encoder + LAG9', f'{MAIN}/readout_combo1_strict_qc_none', 'combo_nosess', 0.198),
       ('Final model', f'{MAIN}/readout_combo5_strict_qc_none', 'combo_nosess', 0.204)]
D = {lab: load_dir(d) for lab, d, _, _ in SRC}
subs = sorted(set().union(*[set(v) for v in D.values()]))
emit('## Table S11a (primary level, strict, 20 % calibration)\n')
emit('| Patient | ' + ' | '.join(l for l, *_ in SRC) + ' |')
emit('|---' * (len(SRC) + 1) + '|')
col = {l: [] for l, *_ in SRC}
kinds = set()
for s in subs:
    vals = []
    for lab, _, m, _ in SRC:
        v, k = model_r(D[lab][s], m) if s in D[lab] else (np.nan, 'missing')
        kinds.add(k); col[lab].append(v); vals.append(f'{v:.3f}')
    emit(f'| {s} | ' + ' | '.join(vals) + ' |')
emit('| **Mean** | ' + ' | '.join(f'**{np.nanmean(col[l]):.3f}**' for l, *_ in SRC) + ' |\n')
emit(f'- nguồn số: {sorted(kinds)}')
for lab, _, _, exp in SRC:
    check(lab, np.nanmean(col[lab]), exp)

In [ ]:
#@title 7. Bảng S11c: theo bệnh nhân, ScaleOLDNEW (primary, strict)
P17 = f'{RUNS}/p17'
SRC = [('Base', f'{P17}/main/final_primary', 'base', 0.206),
       ('Fixed-1', f'{P17}/no_env/fixed1_primary', 'combo_nosess', 0.236),
       ('Fixed-3', f'{P17}/no_env/fixed3_primary', 'combo_nosess', 0.235),
       ('Learned-3', f'{P17}/no_env/learned3_primary', 'combo_nosess', 0.242),
       ('Final', f'{P17}/main/final_primary', 'combo_nosess', 0.242)]
D = {lab: load_dir(d) for lab, d, _, _ in SRC}
subs = sorted(set().union(*[set(v) for v in D.values()]))
emit('## Table S11c (ScaleOLDNEW, primary level, strict, 20 % calibration)\n')
emit('| Patient | ' + ' | '.join(l for l, *_ in SRC) + ' |')
emit('|---' * (len(SRC) + 1) + '|')
col = {l: [] for l, *_ in SRC}
for s in subs:
    vals = []
    for lab, _, m, _ in SRC:
        v, _k = model_r(D[lab][s], m) if s in D[lab] else (np.nan, 'missing')
        col[lab].append(v); vals.append(f'{v:.3f}')
    emit(f'| {s} | ' + ' | '.join(vals) + ' |')
emit('| **Mean** | ' + ' | '.join(f'**{np.nanmean(col[l]):.3f}**' for l, *_ in SRC) + ' |\n')
for lab, _, _, exp in SRC:
    check(lab, np.nanmean(col[lab]), exp)
p = f'{P17}/p17_report.md'
emit('\n- p17_report.md: ' + ('có' if os.path.exists(p) else '**THIẾU**') + '\n')

In [ ]:
#@title 8. Encoder short: 0,203 hay 0,204? (secondary, transductive, ridge một encoder, 20 %)
emit('## Encoder ablations (fold_*.json of Phase 2/3C)\n')
emit('| Run | Folds | enc_ridge | enc_ridge_cal | enc0_ridge | base |')
emit('|---|---|---|---|---|---|')
for name in ('shared_bip_car', 'shared_seed1', 'shared_seed2', 'shared_short', 'shared_no_env'):
    F = [json.load(open(f)) for f in sorted(glob.glob(f'{RUNS}/{name}/fold_sub-*.json'))]
    if not F:
        emit(f'| {name} | 0 | – | – | – | – |'); continue
    def mean(k):
        v = [f['fracs']['0.2']['mean'].get(k, np.nan) for f in F if '0.2' in f['fracs']]
        return f'{np.nanmean(v):.4f}' if v else '–'
    emit(f"| {name} | {len(F)} | {mean('enc_ridge')} | {mean('enc_ridge_cal')} | {mean('enc0_ridge')} | {mean('base')} |")
emit('\n- Bản thảo v10.4: short 0.203, no-env 0.203, main 0.206; output 3C trước đây: short 0.204.\n')
print('\nĐã ghi:', REPORT)